In [ ]:
%%writefile ambiente.h
// Contiene: i parametri, gli ostacoli (ostacoli e fori), la maschera della griglia,
// le sorgenti della passata 1 (bordo degli ostacoli)
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>
#include <algorithm>

// Forme con cui si descrive l'ambiente (coordinate nel cubo [-1, 1]^3)
struct Punto   { double x, y, z; };
struct Zona { double x0, x1, y0, y1, z0, z1; };       // parallelepipedo: x0 <= x <= x1, y0 <= y <= y1, z0 <= z <= z1


// AMBIENTE: per cambiarlo si modifica solo questa parte
const double MARGINE = 0.06;                             // distanza minima dagli ostacoli (raggio del drone + sicurezza)
const double D_SICUREZZA = 0.3;                          // distanza dagli ostacoli oltre la quale si va a velocita' piena
const double V_MIN = 0.1;                                // velocita' minima, a ridosso degli ostacoli (> 0)

const Punto A = { -0.8,  0.0, -0.6 };                    // partenza del drone: primo scomparto, in basso
const Punto B = {  0.85, 0.45,  0.3 };                   // arrivo (sorgente del FIM): terzo scomparto, lontano dai muri

// Cubo 3D: [-1, 1]^3
const double SP = 0.04;                                  // spessore delle pareti della stanza
const std::vector<Zona> PARETI = {                      // le 6 pareti della stanza (pavimento e soffitto compresi)
    { -1.0, -1.0 + SP, -1.0, 1.0, -1.0, 1.0 }, { 1.0 - SP, 1.0, -1.0, 1.0, -1.0, 1.0 },         // lungo x
    { -1.0, 1.0, -1.0, -1.0 + SP, -1.0, 1.0 }, { -1.0, 1.0, 1.0 - SP, 1.0, -1.0, 1.0 },         // lungo y
    { -1.0, 1.0, -1.0, 1.0, -1.0, -1.0 + SP }, { -1.0, 1.0, -1.0, 1.0, 1.0 - SP, 1.0 },         // lungo z
};
// Tre scomparti separati da due muri; ogni muro ha una porta centrale (che si chiude durante il volo) e una finestra
const Zona MURO_1 = { -0.45, -0.35, -1.00, 1.00, -1.00, 1.00 };          // primo muro (x = -0.4)
const Zona MURO_2 = { -0.05, 0.05, -1.00, 1.00, -1.00, 1.00 };          // secondo muro (x = 0)
const Zona PORTA_1    = { -0.45, -0.35, -0.25, 0.25, -1.00, 0.10 };   // porta centrale del primo muro (dal pavimento)
const Zona FINESTRA_1 = { -0.45, -0.35, 0.45, 0.75, 0.30, 0.70 };   // finestra del primo muro: in alto, verso y > 0
const Zona PORTA_2    = { -0.05, 0.05, -0.25, 0.25, -1.00, 0.10 };   // porta centrale del secondo muro
const Zona FINESTRA_2 = { -0.05, 0.05, -0.75, -0.45, -0.55, -0.10 }; // finestra del secondo muro: accanto alla porta 2, alla stessa altezza, verso y < 0

// Ripianificazione: la mappa cambia durante il volo. imposta_mappa(m) sceglie la mappa attuale:
// 0 = tutte le porte aperte, 1 = porta 1 chiusa, 2 = porte 1 e 2 chiuse (restano le due finestre)
inline std::vector<Zona> OSTACOLI;
inline std::vector<Zona> FORI;                        // zone in cui l'ostacolo viene tolto (porte e finestre aperte)
inline void imposta_mappa(int m)
{
    OSTACOLI = PARETI;
    OSTACOLI.push_back(MURO_1);                          // is a built-in member function used to add a new element of a dynamic container
    OSTACOLI.push_back(MURO_2);
    FORI = { FINESTRA_1, FINESTRA_2 };                   // le finestre restano sempre aperte
    if (m < 1) FORI.push_back(PORTA_1);                  // la porta 1 e' aperta solo nella mappa 0
    if (m < 2) FORI.push_back(PORTA_2);                  // la porta 2 e' aperta nelle mappe 0 e 1
}
inline const bool MAPPA_INIZIALE = (imposta_mappa(0), true);   // all'avvio: mappa 0

// Vero se il punto (x, y, z) e' dentro la ostacolo os (bordi compresi)
inline bool dentro(const Zona& os, double x, double y, double z)
{
    return os.x0 <= x && x <= os.x1 && os.y0 <= y && y <= os.y1 && os.z0 <= z && z <= os.z1;
}

// Vero se il punto (x, y, z) e' dentro un ostacolo (i FORI tolgono l'ostacolo)
inline bool ostacolo(double x, double y, double z)
{
    for (const auto& f : FORI)    if (dentro(f, x, y, z)) return false;
    for (const auto& os : OSTACOLI) if (dentro(os, x, y, z)) return true;
    return false;
}

// Maschera della griglia: libero = 1, ostacolo = 0 (resta INF)
inline std::vector<unsigned char> costruisci_maschera(int N)
{
    const double h = 2.0 / (N - 1);
    const int n = N * N * N;
    std::vector<unsigned char> mask(n);
    for (int q = 0; q < n; ++q) {                     // un nodo alla volta
    const int j = q % N, i = (q / N) % N, k = q / (N * N);
    mask[q] = ostacolo(-1.0 + j * h, -1.0 + i * h, -1.0 + k * h) ? 0 : 1;
    }
    // Margine: diventa ostacolo ogni nodo entro MARGINE da un ostacolo
    const int r = (int)std::ceil(MARGINE / h);               // raggio sfera intorno all'ostacolo
    std::vector<unsigned char> margin_mask = mask;
    for (int q = 0; q < n; ++q) {
        if (mask[q]) continue;                                   
    const int k = q / (N * N), i = (q / N) % N, j = q % N;
    const bool bordo = (j > 0 && mask[q - 1]) || (j < N - 1 && mask[q + 1]) ||          // non usciamo dalla griglia e che toccano un nodo libero
                       (i > 0 && mask[q - N]) || (i < N - 1 && mask[q + N]) ||
                       (k > 0 && mask[q - N * N]) || (k < N - 1 && mask[q + N * N]);
    if (!bordo) continue;
    for (int dk = -r; dk <= r; ++dk)                          // tutti i nodi entro la sferadi raggio MARGINE
        for (int di = -r; di <= r; ++di)
            for (int dj = -r; dj <= r; ++dj) {
                const int kk = k + dk, ii = i + di, jj = j + dj;
                if (kk < 0 || kk >= N || ii < 0 || ii >= N || jj < 0 || jj >= N) continue;
                if ((dk * dk + di * di + dj * dj) * h * h <= MARGINE * MARGINE)
                    margin_mask[(kk * N + ii) * N + jj] = 0;
            }
}
return margin_mask;
}
// Sorgenti della passata 1: i nodi liberi che toccano un ostacolo
inline std::vector<int> nodi_di_bordo(const std::vector<unsigned char>& mask, int N)
{
    std::vector<int> bordo;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
                const int q = (k * N + i) * N + j;
                if (!mask[q]) continue;                                  // solo nodi liberi...
                if ((j > 0 && !mask[q - 1]) || (j < N - 1 && !mask[q + 1]) ||
                    (i > 0 && !mask[q - N]) || (i < N - 1 && !mask[q + N]) ||
                    (k > 0 && !mask[q - N * N]) || (k < N - 1 && !mask[q + N * N]))
                    bordo.push_back(q);                                  // ...che hanno un vicino ostacolo
            }
    return bordo;
}

// FM2: dalla distanza d dagli ostacoli alla lentezza F = 1/v della passata 2
// v = V_MIN + (1 - V_MIN) * min(d / D_SICUREZZA, 1): F = 10 sul bordo, 1 lontano
inline std::vector<float> lentezza_fm2(const std::vector<float>& d)
{
    std::vector<float> F(d.size());
    for (size_t q = 0; q < d.size(); ++q) {
        const double r = std::min((double)d[q] / D_SICUREZZA, 1.0);       // distanza normalizzata: 0 sul bordo, 1 oltre D_SICUREZZA (o se d = INF)
        F[q] = (float)(1.0 / (V_MIN + (1.0 - V_MIN) * r));
    }
    return F;
}
// Controllo iniziale: A e B non devono stare dentro un ostacolo
inline bool ambiente_valido()
{
    return !ostacolo(A.x, A.y, A.z) && !ostacolo(B.x, B.y, B.z);
}

// Dal punto (x, y, z) all'indice del nodo piu' vicino (serve per A e B)
int nodo(int N, double x, double y, double z){
    const double h = 2.0 / (N - 1);
    const int j = (int)lround((x + 1.0) / h), i = (int)lround((y + 1.0) / h), k = (int)lround((z + 1.0) / h);
    return (k * N + i) * N + j;
}

In [ ]:
%%writefile godunov3d.h
#pragma once           // il file viene incluso una sola volta
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <cmath>       // fminf, sqrtf

const float INF = 1e30f;                 // "infinito": nodo non ancora raggiunto dall'onda
// Risolutore locale 3D (Godunov, primo ordine), comune ai tre metodi.
// __host__ __device__: compilata per la CPU (la usa il FMM) e per la GPU (Block FIM e FIM-Jacobi)
__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;                              // tempo per attraversare una cella
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

// Controllo degli errori CUDA: se una chiamata fallisce, stampa dove e perche' ed esce
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

In [ ]:
%%writefile fmm3d.h
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include "godunov3d.h"  // risolutore locale

typedef std::pair<float, int> Entry;     // coppia (u, indice del nodo) per la coda di priorita' del FMM

// Nuovo u del nodo (k, i, j) per il FMM: minimo dei vicini per asse + risolutore locale
float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int k, int i, int j, int N, float h)
{
    int idx = (k * N + i) * N + j;  // indice globale del nodo (k, i, j)
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[idx - 1] : INF;
    float right = (j < N - 1) ? u[idx + 1] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[idx - N] : INF;
    float up    = (i < N - 1) ? u[idx + N] : INF;
    // c: min dei vicini lungo z (stessa colonna e riga, piano k-1 e k+1)
    float back  = (k > 0)     ? u[(idx - N * N)] : INF;
    float front = (k < N - 1) ? u[(idx + N *N)] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);
    float c = fminf(back, front);

    float q = solve_quadratic3(a, b, c, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ijk, u)
}
// Fast Marching Method (Alg. 3) su CPU: il riferimento per il Block FIM
// Fissa i nodi in ordine di tempo crescente con una coda di priorita'
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, const std::vector<int>& srcs, const std::vector<unsigned char>& mask)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N * N, LONTANO);    // array di char NxNxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    for (int s : srcs){heap.push(Entry(u[s], s)); state[s] = ATTIVO;}

    const int dk[6] = {  0, 0,  0, 0, -1, 1 };       // spostamenti verso i 6 vicini:
    const int di[6] = {  0, 0, -1, 1,  0, 0 };       // sinistra, destra, sotto, sopra,
    const int dj[6] = { -1, 1,  0, 0,  0, 0 };       // davanti, dietro

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int k = x / (N*N), i = (x / N) % N, j = x % N;                    // strato, riga, colonna del nodo Trial
        for (int d = 0; d < 6; ++d) {                // foreach x_nb adiacente a Trial
            int ni = i + di[d], nj = j + dj[d], nk = k + dk[d];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N || nk < 0 || nk >= N) continue;   // fuori dal dominio
            int nb = (nk * N + ni) * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO || !mask[nb]) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, nk, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}

In [ ]:
%%writefile block_fim3d.cuh
#pragma once           // il file viene incluso una sola volta
// Block FIM 3D (Alg. 7) sulla GPU: blocchi 8^3 in shared memory, uscita anticipata, maschera degli ostacoli
#include <cstdio>
#include <cstdlib>
#include <vector>
#include <algorithm>       // std::min
#include <cuda_runtime.h>
#include "godunov3d.h"  // risolutore locale

#define BLOCK 8  // blocco 8 x 8 x 8 = 512 nodi, un thread per nodo
// Risolutore locale in shared memory: i 6 vicini del nodo sono nella copia s (blocco + halo)
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

// Aggiorna i blocchi di una lista (Alg. 7): lista dei blocchi attivi (Step 1, niter iterazioni) 
// oppure dei candidati (Step 2, 1 iterazione)
__global__ void update_blocks(const int* list, const int* quantity, int niter, float* u, const float* F,
                             int N, int nbx, float h, const unsigned char* mask, float eps, unsigned char* Cb_out, int* is_candidate)
{
    // quantity = 0/1; *quantity = numbero di blocchi attivi
    if (quantity && blockIdx.x >= *quantity) return;
    // (es. N = 33) giro: 1 -> blocchi CUDA lanciati: 1 -> blockIdx.x: 0 -> block = active_list[blockIdx.x]: 62
    // (es. N = 33) giro: 3 -> blocchi CUDA lanciati: 6 -> blockIdx.x: 0, 1, ..., 5 -> block = active_list[blockIdx.x]: 61, 63, 57, 67, 37, 87
    const int block = list[blockIdx.x];

    // Allocazione statica di shared memory
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];              
    __shared__ bool s_Cn[BLOCK * BLOCK * BLOCK];                  // i flag Cn dei 512 nodi
    // posizione del thread nel suo blocco CUDA: ognuno da 0 a 7
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    // bj: asse x , bi: asse y e bk: asse z; identifica blocco nella griglia
    int bj = block % nbx, bi = (block / nbx) % nbx, bk = block / (nbx * nbx); 
    // numero del thread dentro il blocco; 
    int tid = (tz*BLOCK + ty) * BLOCK + tx; // 0, ..., 511
    // posizione globale rispetto x,y e z: lungo tutta la griglia
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;
    int idx = (k * N + i) * N + j;  // 0, ..., N^3-1
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || !mask[idx];             // nodo da non aggiornare: fuori dalla griglia o maschera
    float f = inside ? F[idx] : 1.0f;
    // Global -> Shared
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        // posizione nella shared memory
        int sz = m / ((BLOCK + 2) * (BLOCK + 2)), sy = (m / (BLOCK + 2)) % (BLOCK + 2), sx = m % (BLOCK + 2);
        // coordinate globali della cella (sz, sy, sx): -1 perché c'è l'halo                          
        int gi = bi * BLOCK + sy - 1, gj = bj * BLOCK + sx - 1, gk = bk * BLOCK + sz - 1;
        
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();
    bool conv = true;
    for (int it = 0; it < niter; ++it){
        // p: valore del nodo all'inizio di questa iterazione
        float p = s_u[tz+1][ty+1][tx+1];
        // q: nuovo valore calcolato in questa iterazione
        float q = fixed ? p : solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);  
        __syncthreads();
        s_u[tz+1][ty + 1][tx + 1] = q;
        conv = fabsf(p - q) < eps;
        if (!__syncthreads_or(!conv)) break;   // tutti hanno scritto; se nessun nodo e' cambiato, esci
    }
    // Shared -> Global: ogni thread riscrive il proprio nodo
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];
    // Cn del nodo: vero se non è cambiato nell'ultima iterazione
    s_Cn[tid] = conv;
    __syncthreads();
    for (int reduct = BLOCK * BLOCK * BLOCK / 2; reduct > 0; reduct /= 2) {     // 256, ..., 4, 2, 1
        if (tid < reduct) s_Cn[tid] = s_Cn[tid] && s_Cn[tid + reduct];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_Cn[0];
    // solo candidati: pronto per il giro dopo
    if (is_candidate && tid == 0) is_candidate[block] = 0;  
}

// costruisce la lista A dei blocchi attivi (Cb = 0)
__global__ void build_active_list(const unsigned char* Cb, unsigned char* in_list,
                                  int* active_list, int* n_active, int n_blocks) 
{
    // blockIdx: blocco nel lancio in corso; blockDim (monodimensionale CUDA): 256; threadIdx:0,...,255
    int b = blockIdx.x * blockDim.x + threadIdx.x;           // numero del thread(blocco) su tutto il lancio
    // N = 65 -> nbx = 9 -> n_blocks = 729 -> list_blocks = 3 -> thread lanciati = 768 -> thread in eccesso = 39 
    if (b >= n_blocks) return;                               // thread(blocco) in eccesso lanciati da CUDA                                   
    in_list[b] = !Cb[b];                                      // fotografia della lista A di questo giro
    if (!Cb[b]) active_list[atomicAdd(n_active, 1)] = b;               // se NON convergente aggiungi thread(blocco) alla lista A
}
// Step 2: i 6 vicini dei blocchi appena convergenti diventano candidati
__global__ void build_candidate_list(const int* active_list, int n_active, int nbx, const unsigned char* in_list,
                                     const unsigned char* Cb_step1, unsigned char* Cb,
                                     int* is_candidate, int* cand_list, int* cand_count)
{
    int pos = blockIdx.x * blockDim.x + threadIdx.x;
    // se è maggiore o uguale al numero di blocchi attivi fai return
    if (pos >= n_active) return;                       
    int b = active_list[pos];         // un blocco della lista A; N = 33 -> A ={61, 63, ...,87}; active_list[0] =61
    Cb[b] = Cb_step1[b];             // leggi il valore di Cb
    if (!Cb_step1[b]) return;        // proseguono solo i blocchi appena convergenti
    // bj: blocco lungo x; bi: blocco lungo y; bk: blocco lungo z
    int bj = b % nbx, bi = (b / nbx) % nbx, bk = b / (nbx * nbx);
    // sinistra, destra, sotto, sopra e davanti, dietro
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};
    // verifico se si trova nella griglia
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // scarta blocchi fuori dalla griglia
        // scarta blocchi attivi e non convergenti
        if (in_list[nb[d]] && !Cb_step1[nb[d]]) continue;          // d: indica la direzione; N = 33; nb[d] = [61, 63, 57, 67, 37, 87];      
        if (atomicExch(&is_candidate[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}
// Ogni giro: lista A -> Step 1 -> Step 2; fine quando A e' vuota
int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, int niter, float* ms)
{
    // nbx: numero di blocchi per lato; n_blocks: blocchi totali
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx;
    // bytes: dimensione totale griglia 3D
    const size_t bytes = (size_t)N * N * N * sizeof(float);
    // Cb: si inizializza vettore di n_blocks valori a 1
    std::vector<unsigned char> Cb(n_blocks, 1);
    // strato, riga e colonna del nodo sorgente
    for (int s : srcs){
    const int sk = s / (N * N), si = (s / N) % N, sj = s % N;
    // Cb: vale 1 tranne nel blocco della sorgente (0) 
    Cb[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;
    }
    // Memoria sulla GPU e copia dei dati iniziali (CPU -> GPU)
    float *d_u, *d_F; unsigned char *d_Cb, *d_Cb_step1, *d_in_list, *d_mask; int *d_active_list, *d_n_active;
    int *d_is_candidate, *d_cand_list, *d_cand_count;
    CUDA_CHECK(cudaMalloc(&d_u, bytes)); CUDA_CHECK(cudaMalloc(&d_F, bytes)); CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb_step1, n_blocks)); CUDA_CHECK(cudaMalloc(&d_in_list, n_blocks)); CUDA_CHECK(cudaMalloc(&d_active_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_n_active, sizeof(int))); CUDA_CHECK(cudaMalloc(&d_mask, N*N*N));
    // Memoria per i blocchi candidati
    CUDA_CHECK(cudaMalloc(&d_is_candidate, n_blocks * sizeof(int))); CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));
    // si copiano d_u, d_F e d_Cb dalla CPU alla GPU
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice)); CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb.data(), n_blocks, cudaMemcpyHostToDevice)); CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), N*N*N, cudaMemcpyHostToDevice));
    // si azzerano d_Cb_step1 e d_is_candidate
    CUDA_CHECK(cudaMemset(d_Cb_step1, 0, n_blocks)); CUDA_CHECK(cudaMemset(d_is_candidate, 0, n_blocks * sizeof(int)));
    // list_blocks: blocchi CUDA da 256 threads per avere un thread per ogni blocco di nodi 
    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;
    // si misura l'ora prima del ciclo
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start)); CUDA_CHECK(cudaEventCreate(&stop)); CUDA_CHECK(cudaEventRecord(start));                     
    int iter = 0;
    while (true) {
        // azzera il numero di blocchi attivi                                           
        int n_active = 0;  // contatore su CPU
        CUDA_CHECK(cudaMemset(d_n_active, 0, sizeof(int)));    // contatore su GPU
        // lancia list_blocks CUDA da 256 threads ciascuno, sufficiente per avere un thread ogni blocco
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_in_list, d_active_list, d_n_active, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        // aggiorna n_active con d_n_active
        CUDA_CHECK(cudaMemcpy(&n_active, d_n_active, sizeof(int), cudaMemcpyDeviceToHost));
        if (n_active == 0) break;  // se n_active == 0, lista A vuota, termina ciclo
        ++iter;                 // conta i giri
        // Step 1: foreach b in A in parallelo; lancia n_active blocchi 8x8x8
        update_blocks<<<n_active, dim3(BLOCK, BLOCK, BLOCK)>>>(d_active_list, nullptr, niter, d_u, d_F, N, nbx, h, d_mask, eps, d_Cb_step1, nullptr);
        CUDA_CHECK(cudaGetLastError());
        // azzeriamo il contatore dei candidati sulla GPU
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        // lancia n_active blocchi CUDA da 256 threads
        build_candidate_list<<<(n_active + threads-1) / threads, threads>>>(d_active_list, n_active, nbx, d_in_list, d_Cb_step1, d_Cb,
                                                                          d_is_candidate, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        // ogni blocco ha al più 6 vicini, quindi 6*n_active
        update_blocks<<<std::min(6 * n_active, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, 1,
                                                                                    d_u, d_F, N, nbx, h, d_mask, eps, d_Cb, d_is_candidate);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start)); CUDA_CHECK(cudaEventDestroy(stop));
    // Risultato sulla CPU (GPU -> CPU) e liberazione della memoria
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb_step1)); CUDA_CHECK(cudaFree(d_mask));
    CUDA_CHECK(cudaFree(d_in_list)); CUDA_CHECK(cudaFree(d_active_list)); CUDA_CHECK(cudaFree(d_n_active));
    CUDA_CHECK(cudaFree(d_is_candidate)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}


In [ ]:
%%writefile jacobi_fim.cuh
// FIM-Jacobi senza lista di G. Cirillo, riscritto in CUDA 3D
// i nodi si aggiornano con Godunov usando i valori dell'iterazione prima (Jacobi), nuovo = min(vecchio, Godunov) come il
#pragma once
#include <vector>
#include <utility>          // std::swap
#include <cuda_runtime.h>
#include "godunov3d.h"  // risolutore locale

const int CONTROLLO = 25;   // ogni quante iterazioni si controlla la convergenza (come check_every del suo codice)

// Un thread per nodo: nuovo valore del nodo x dai 6 vicini dell'iterazione precedente
__global__ void jacobi_update(const float* cur, const float* F, const unsigned char* mask, float* nxt,
                              int N, float h, float eps, int* cambiato)
{
    const int x = blockIdx.x * blockDim.x + threadIdx.x;
    if (x >= N * N * N) return;                                   // thread in eccesso
    const float p = cur[x];
    if (p == 0.0f || !mask[x]) { nxt[x] = p; return; }            // sorgente o ostacolo: non cambia (gli ostacoli restano a INF)
    const int j = x % N, i = (x / N) % N, k = x / (N * N);
    const float a = fminf(j > 0 ? cur[x - 1] : INF, j < N - 1 ? cur[x + 1] : INF);              // vicini lungo x
    const float b = fminf(i > 0 ? cur[x - N] : INF, i < N - 1 ? cur[x + N] : INF);              // vicini lungo y
    const float c = fminf(k > 0 ? cur[x - N * N] : INF, k < N - 1 ? cur[x + N * N] : INF);      // vicini lungo z
    const float q = fminf(p, solve_quadratic3(a, b, c, F[x], h));                               // min(vecchio, Godunov)
    nxt[x] = q;
    if (cambiato && p - q >= eps) *cambiato = 1;                  // solo nelle iterazioni di controllo: qualcuno cambia ancora
}

// Stessa interfaccia del FIM a nodi: le sorgenti sono gia' a 0 in u (srcs non serve qui). Restituisce le iterazioni;
// in ms il tempo del solo ciclo (eventi CUDA), come per il Block FIM
int jacobi_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
               const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, float* ms)
{
    (void)srcs;
    const int n = N * N * N;
    const size_t bytes = (size_t)n * sizeof(float);
    float *d_cur, *d_nxt, *d_F; unsigned char* d_mask; int* d_cambiato;
    CUDA_CHECK(cudaMalloc(&d_cur, bytes)); CUDA_CHECK(cudaMalloc(&d_nxt, bytes)); CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_mask, n)); CUDA_CHECK(cudaMalloc(&d_cambiato, sizeof(int)));
    CUDA_CHECK(cudaMemcpy(d_cur, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), n, cudaMemcpyHostToDevice));
    const int threads = 256, blocchi = (n + threads - 1) / threads;   // come il suo threads_per_block = 256
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start)); CUDA_CHECK(cudaEventCreate(&stop)); CUDA_CHECK(cudaEventRecord(start));
    int it = 0;
    while (true) {
        const bool controlla = (it % CONTROLLO == 0);
        if (controlla) CUDA_CHECK(cudaMemset(d_cambiato, 0, sizeof(int)));
        jacobi_update<<<blocchi, threads>>>(d_cur, d_F, d_mask, d_nxt, N, h, eps, controlla ? d_cambiato : nullptr);
        CUDA_CHECK(cudaGetLastError());
        std::swap(d_cur, d_nxt);                                  // il nuovo diventa il vecchio
        ++it;
        if (controlla) {                                          // sincronizzazione GPU -> CPU solo ogni CONTROLLO iterazioni
            int cambiato = 0;
            CUDA_CHECK(cudaMemcpy(&cambiato, d_cambiato, sizeof(int), cudaMemcpyDeviceToHost));
            if (!cambiato) break;
        }
    }
    CUDA_CHECK(cudaEventRecord(stop)); CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));
    CUDA_CHECK(cudaEventDestroy(start)); CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaMemcpy(u.data(), d_cur, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_cur)); CUDA_CHECK(cudaFree(d_nxt)); CUDA_CHECK(cudaFree(d_F));
    CUDA_CHECK(cudaFree(d_mask)); CUDA_CHECK(cudaFree(d_cambiato));
    return it;
}

In [ ]:
%%writefile main.cu
// FM2 (passata 1 + passata 2) con il FMM, il FIM-Jacobi e il Block FIM: tabella al variare di N e ripianificazione nel labirinto
#include <cstdio>
#include <chrono>          // tempi sulla CPU
#include <fstream>         // salvataggio su file
#include <string>          // nomi dei file delle mappe
#include "fmm3d.h"         // Fast Marching (CPU)
#include "ambiente.h"      // stanza, ostacoli, maschera, lentezza del FM2
#include "block_fim3d.cuh" // Block FIM (GPU)
#include "jacobi_fim.cuh"  // FIM-Jacobi (GPU)


// FM2 completo (passata 1 + passata 2) sulla mappa attuale, con il FMM (CPU), il FIM-Jacobi (GPU) o il Block FIM (GPU).
// Restituisce u (tempo di arrivo da B), la F del FM2 e il tempo di calcolo delle due passate [ms];
// come valore di ritorno le iterazioni della passata 2 (giri del Block FIM, iterazioni del FIM-Jacobi, 0 per il FMM).
enum Metodo { FMM, FIM_JACOBI, BLOCK_FIM };                  // con quale metodo si risolvono le due passate
int fm2(Metodo metodo, int Nk, const std::vector<unsigned char>& mask, float eps, int niter,
         std::vector<float>& u, std::vector<float>& Fk, double& t_ms)
{
    const float hk = 2.0f / (Nk - 1);
    const int n = Nk * Nk * Nk;
    // prima passata: numero di nodi liberi che toccano un ostacolo
    const std::vector<int> bordo = nodi_di_bordo(mask, Nk);
    // seconda passata: percorso minimo per B          
    const std::vector<int> srcs = { nodo(Nk, B.x, B.y, B.z) };          
    std::vector<float> d(n, INF);
    for (int s : bordo) d[s] = 0.0f;
    const std::vector<float> uno(n, 1.0f);
    u.assign(n, INF);
    for (int s : srcs) u[s] = 0.0f;
    int giri = 0;
    if (metodo == BLOCK_FIM) {                               // GPU: tempo del solo ciclo dei giri (come nelle tabelle)
        float ms1 = 0.0f, ms2 = 0.0f;
        // calcolo la distanza dei nodi liberi dagli ostacoli
        block_fim(d, uno, Nk, hk, bordo, mask, eps, niter, &ms1);
        // definisco la lentezza
        Fk = lentezza_fm2(d);
        // tempo di arrivo in B
        giri = block_fim(u, Fk, Nk, hk, srcs, mask, eps, niter, &ms2);
        // tempo impiegato dal FM2 con Block FIM
        t_ms = ms1 + ms2;
    } else if (metodo == FIM_JACOBI) {                         // GPU: le due passate con il FIM-Jacobi (tempo del solo ciclo)
        float ms1 = 0.0f, ms2 = 0.0f;
        jacobi_fim(d, uno, Nk, hk, bordo, mask, eps, &ms1);
        Fk = lentezza_fm2(d);
        giri = jacobi_fim(u, Fk, Nk, hk, srcs, mask, eps, &ms2);
        t_ms = ms1 + ms2;
    } else {                                                 // CPU: le due chiamate al FMM
        auto t0 = std::chrono::high_resolution_clock::now();
        fmm(d, uno, Nk, hk, bordo, mask);
        Fk = lentezza_fm2(d);
        fmm(u, Fk, Nk, hk, srcs, mask);
        auto t1 = std::chrono::high_resolution_clock::now();
        t_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
    }
    return giri;
}

// Scrive i file di una mappa per Python: u, maschera, F e ambiente (ostacoli e fori della mappa attuale)
void scrivi_mappa(int Nk, int m, const std::vector<float>& u, const std::vector<unsigned char>& mask, const std::vector<float>& Fk)
{
    const std::string t = std::to_string(Nk) + "_mappa" + std::to_string(m);
    std::ofstream out("fim_" + t + ".bin", std::ios::binary);
    out.write(reinterpret_cast<const char*>(u.data()), u.size() * sizeof(float));
    std::ofstream om("mask_" + t + ".bin", std::ios::binary);
    om.write(reinterpret_cast<const char*>(mask.data()), mask.size());
    std::ofstream of("F_" + t + ".bin", std::ios::binary);
    of.write(reinterpret_cast<const char*>(Fk.data()), Fk.size() * sizeof(float));
    std::ofstream amb("ambiente_mappa" + std::to_string(m) + ".txt");
    amb.precision(15);
    amb << "MARGINE " << MARGINE << "\n";
    amb << "A " << A.x << " " << A.y << " " << A.z << "\n";
    amb << "B " << B.x << " " << B.y << " " << B.z << "\n";
    for (const auto& os : OSTACOLI) amb << "ZONA " << os.x0 << " " << os.x1 << " " << os.y0 << " " << os.y1 << " " << os.z0 << " " << os.z1 << "\n";
    for (const auto& f : FORI)    amb << "FORO " << f.x0 << " " << f.x1 << " " << f.y0 << " " << f.y1 << " " << f.z0 << " " << f.z1 << "\n";
}

int main()
{
    // Informazioni sulla GPU usata
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d | max thread per blocco: %d | shared memory per blocco: %zu KB\n",
           prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock, prop.sharedMemPerBlock / 1024);
    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza
    const int niter = 8;                                     // iterazioni locali massime (uscita anticipata): la migliore con gli ostacoli (prove 12 / 24)

    if (!ambiente_valido()) {                                // A o B dentro un ostacolo: il problema non ha senso
        printf("Errore: A o B sono dentro un ostacolo, correggi ambiente.h\n");
        return 1;
    }
    printf("speedup = t metodo / t Block FIM; max|...| = differenza massima dalla u del Block FIM\n");
    printf("    N   giri BFIM   iter Jac   max|FMM-BFIM|   max|Jac-BFIM|   u(A) BFIM   u(A) FMM   t FMM [ms]   t Jac [ms]   t BFIM [ms]   sp vs FMM   sp vs Jac\n");

    for (int Nk : {193, 257, 385 }) {
        const int ripetizioni = 3;                           // ogni tempo = la media di 3 misure...
        const int ripetizioni_fmm = (Nk >= 385) ? 1 : 3;     // ...tranne il FMM a N = 385: dura minuti e varia di pochi percento
        const std::vector<unsigned char> mask = costruisci_maschera(Nk);    // 1 = libero, 0 = ostacolo
        const int iA = nodo(Nk, A.x, A.y, A.z);              // nodo della partenza A
        // FM2 completo (passata 1 + passata 2) con i tre metodi: ognuno usa la propria F
        std::vector<float> u_fmm, u_bfim, u_jac, F_fmm, F_bfim, F_jac;
        double t_fmm = 0.0, t_bfim = 0.0, t_jac = 0.0;
        int it = 0, it_jac = 0;                              // giri del Block FIM e iterazioni del FIM-Jacobi (passata 2)
        for (int r = 0; r < ripetizioni; ++r) {
            double t;
            if (r < ripetizioni_fmm) { fm2(FMM, Nk, mask, eps, niter, u_fmm, F_fmm, t); t_fmm += t; }   // il FMM: 1 sola misura a N = 385
            it_jac = fm2(FIM_JACOBI, Nk, mask, eps, niter, u_jac, F_jac, t); t_jac += t;
            it = fm2(BLOCK_FIM, Nk, mask, eps, niter, u_bfim, F_bfim, t); t_bfim += t;
        }
        t_jac /= ripetizioni; t_bfim /= ripetizioni;         // media delle misure
        t_fmm /= ripetizioni_fmm;                            // media delle misure del FMM

        // Controlli: stessa u dei tre metodi, nessun NaN, ostacoli a INF
        float diff = 0.0f, diff_jac = 0.0f;
        int n_nan = 0, ost_non_inf = 0;
        for (size_t q = 0; q < u_bfim.size(); ++q) {
            diff = fmaxf(diff, fabsf(u_fmm[q] - u_bfim[q]));
            diff_jac = fmaxf(diff_jac, fabsf(u_jac[q] - u_bfim[q]));
            if (std::isnan(u_bfim[q])) ++n_nan;
            if (!mask[q] && u_bfim[q] < INF) ++ost_non_inf;
        }
        if (n_nan > 0 || ost_non_inf > 0)                    // si stampa solo se qualcosa non va
            printf("ATTENZIONE (N = %d): %d valori NaN, %d nodi ostacolo raggiunti dall'onda\n", Nk, n_nan, ost_non_inf);

        if (u_bfim[iA] >= INF)                               // l'onda partita da B non e' mai arrivata ad A
            printf("%5d   %9d   %13.2e   A NON RAGGIUNGIBILE da B\n", Nk, it, diff);
        else
            printf("%5d   %9d   %8d   %13.2e   %13.2e   %9.4f   %8.4f   %10.1f   %10.2f   %11.2f   %9.1f   %9.1f\n",
                   Nk, it, it_jac, diff, diff_jac, u_bfim[iA], u_fmm[iA], t_fmm, t_jac, t_bfim,
                   t_fmm / t_bfim, t_jac / t_bfim);
    }
    // RIPIANIFICAZIONE (N = 385) nel labirinto: la mappa cambia durante il volo
    {
        const int Nr = 385;                                  // griglia della ripianificazione 
        const int ripetizioni = 3;                           // ogni tempo = la media di 3 misure...
        const int ripetizioni_fmm = (Nr >= 385) ? 1 : 3;     // ...tranne il FMM a N = 385: dura minuti e varia di pochi percento
        std::ofstream tr("tempi_ripianificazione_" + std::to_string(Nr) + ".txt");   // una riga per mappa: m, t FMM, t Block FIM, t FIM-Jacobi [ms]
        printf("\nRipianificazione (N = %d): tempo per ricalcolare il FM2 (passata 1 + passata 2) dopo ogni cambio di mappa\n", Nr);
        printf("  mappa                       u(A) FMM   u(A) BFIM   max|FMM-BFIM|   max|Jac-BFIM|   t FMM [ms]   t Jac [ms]   t BFIM [ms]   sp vs FMM   sp vs Jac\n");
        const char* nomi[3] = { "0 porte aperte", "1 porta 1 chiusa", "2 porte 1 e 2 chiuse" };
        for (int m = 0; m < 3; ++m) {
            imposta_mappa(m);
            const std::vector<unsigned char> mask = costruisci_maschera(Nr);
            const int iA = nodo(Nr, A.x, A.y, A.z);
            std::vector<float> u_fmm, u_bfim, u_jac, F_fmm, F_bfim, F_jac;
            double t_fmm = 0.0, t_bfim = 0.0, t_jac = 0.0;
            for (int r = 0; r < ripetizioni; ++r) {
                double t;
                if (r < ripetizioni_fmm) { fm2(FMM, Nr, mask, eps, niter, u_fmm, F_fmm, t); t_fmm += t; }   // il FMM: 1 sola misura a N = 385
                fm2(FIM_JACOBI,  Nr, mask, eps, niter, u_jac, F_jac, t); t_jac += t;
                fm2(BLOCK_FIM, Nr, mask, eps, niter, u_bfim, F_bfim, t); t_bfim += t;
            }
            t_jac /= ripetizioni; t_bfim /= ripetizioni;   // media delle misure
            t_fmm /= ripetizioni_fmm;                          // media delle misure del FMM
            float diff = 0.0f, diff_jac = 0.0f;
            for (size_t q = 0; q < u_bfim.size(); ++q) {
                diff = fmaxf(diff, fabsf(u_fmm[q] - u_bfim[q]));
                diff_jac = fmaxf(diff_jac, fabsf(u_jac[q] - u_bfim[q]));
            }
            printf("  %-26s   %8.4f   %9.4f   %13.2e   %13.2e   %10.1f   %10.2f   %11.2f   %9.1f   %9.1f\n",
                   nomi[m], u_fmm[iA], u_bfim[iA], diff, diff_jac, t_fmm, t_jac, t_bfim, t_fmm / t_bfim, t_jac / t_bfim);
            tr << m << " " << t_fmm << " " << t_bfim << " " << t_jac << "\n";       // in coda il FIM-Jacobi
            scrivi_mappa(Nr, m, u_bfim, mask, F_bfim);       // il drone del Block FIM e quello del FMM usano la stessa u
        }
        imposta_mappa(0);
    }
    return 0;
}

In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_75 main.cu -o fim3d && ./fim3d

: 

In [ ]:
# RIPIANIFICAZIONE NEL LABIRINTO - CALCOLO (usa i file scritti da main.cu): i droni partono insieme da A. Poco prima che arrivino a una porta, la porta si chiude
# (cambio 1: porta del primo muro; cambio 2: porta del secondo muro). A ogni cambio ogni drone ricalcola il FM2
# (tempi veri misurati da fim3d.cu) continuando sul percorso vecchio; davanti alla porta chiusa frena e aspetta. Poi usa le finestre.
# Qui: lettura dei dati, ricostruzione del percorso con RK4 (sezione 2) e volo dei droni.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import RegularGridInterpolator

# 0) PARAMETRI
N = 385                                                             # griglia dei file scritti da main.cu (Nr)
METRI_PER_UNITA = 10.0                                              # il cubo [-1, 1]^3 e' una stanza di 20 m di lato
V_MAX = 2.0                                                         # velocita' massima del drone [m/s], dove F = 1
SEC_PER_U = METRI_PER_UNITA / V_MAX                                 # 1 unita' di u = 5 s di volo
X_CAMBIO = {1: -0.54, 2: -0.16}                                     # la porta m si chiude quando il drone supera x = X_CAMBIO[m] (poco prima della porta)
V_PIENA = 0.95 * V_MAX                                              # "velocita' massima": almeno il 95% di V_MAX
METODI = ["FMM", "FIM-Jacobi", "Block FIM"]                                # da sinistra a destra
h = 2.0 / (N - 1)
x = -1.0 + h * np.arange(N)


# 1) DATI: le tre mappe (u da B, maschera, lentezza F, ostacoli) e i tempi di ricalcolo
def leggi_ambiente(nome):
    amb = {"ZONA": [], "FORO": []}
    for riga in open(nome):
        k, *v = riga.split(); v = [float(t) for t in v]
        if k in ("A", "B"): amb[k] = np.array(v)
        elif k == "MARGINE": amb[k] = v[0]
        else: amb[k].append(tuple(v))
    return amb
mappe = []
for m in range(3):
    t = f"{N}_mappa{m}"
    u = np.fromfile(f"fim_{t}.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)    # u[k, i, j] = u(z, y, x)
    libero = np.fromfile(f"mask_{t}.bin", dtype=np.uint8).reshape(N, N, N) == 1
    F = np.fromfile(f"F_{t}.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)
    mappe.append(dict(u=u, libero=libero, F=F, amb=leggi_ambiente(f"ambiente_mappa{m}.txt"),
                      F_i=RegularGridInterpolator((x, x, x), F), peso=RegularGridInterpolator((x, x, x), libero.astype(float))))
A, B = mappe[0]["amb"]["A"], mappe[0]["amb"]["B"]
tempi = {}                                                          # tempo di ricalcolo [s] del FM2 per mappa e metodo
for riga in open(f"tempi_ripianificazione_{N}.txt"):
    m, t_fmm, t_bfim, t_jac = riga.split()                         # una riga per mappa: m, t FMM, t Block FIM, t FIM-Jacobi [ms]
    tempi[int(m)] = {"FMM": float(t_fmm) / 1000.0, "FIM-Jacobi": float(t_jac) / 1000.0, "Block FIM": float(t_bfim) / 1000.0}

# 2) PERCORSO da un punto qualsiasi su una mappa fino a B: discesa continua lungo -grad u, integrata con Runge-Kutta del 4 ordine.
#    Il percorso e' una curva: segue la direzione in cui u (il tempo per arrivare a B) diminuisce piu' in fretta, cioe' il raggio.
nodo = lambda p: tuple(np.round((np.asarray(p)[::-1] + 1.0) / h).astype(int))   # nodo (k, i, j) piu' vicino al punto p = (x, y, z)
def gradiente(mp, p):
    """Gradiente di u nel punto p = (x, y, z): differenze finite nei nodi liberi, interpolate (trilineare) fra gli 8 nodi vicini.
    Vicino agli ostacoli (u = INF) si usa la differenza da un solo lato; i nodi ostacolo non contano."""
    u, libero = mp["u"], mp["libero"]
    q = (np.asarray(p)[::-1] + 1.0) / h                             # posizione in indici (k, i, j), non intera
    q0 = np.clip(np.floor(q).astype(int), 0, N - 2)
    w = q - q0                                                      # pesi dell'interpolazione trilineare
    g, peso_tot = np.zeros(3), 0.0
    for dk in (0, 1):
        for di in (0, 1):
            for dj in (0, 1):
                n = (q0[0] + dk, q0[1] + di, q0[2] + dj)
                if not libero[n] or u[n] >= 1e29: continue          # nodo ostacolo: non contribuisce
                pw = (w[0] if dk else 1 - w[0]) * (w[1] if di else 1 - w[1]) * (w[2] if dj else 1 - w[2])
                gn = np.zeros(3)                                    # gradiente nel nodo, ordine (z, y, x)
                for a in range(3):
                    su, giu = list(n), list(n); su[a] += 1; giu[a] -= 1
                    su, giu = tuple(su), tuple(giu)
                    ok_su = su[a] < N and libero[su] and u[su] < 1e29
                    ok_giu = giu[a] >= 0 and libero[giu] and u[giu] < 1e29
                    if ok_su and ok_giu: gn[a] = (u[su] - u[giu]) / (2 * h)     # differenza centrata
                    elif ok_su: gn[a] = (u[su] - u[n]) / h                       # solo in avanti
                    elif ok_giu: gn[a] = (u[n] - u[giu]) / h                     # solo all'indietro
                g += pw * gn; peso_tot += pw
    return g[::-1] / peso_tot if peso_tot > 0 else np.zeros(3)     # in ordine (x, y, z)
def percorso(mp, p0):
    """Percorso da p0 a B seguendo dp/ds = -grad u / |grad u| (la direzione di discesa piu' ripida), integrata con RK4 a passo h/2."""
    ds = h / 2                                                      # passo di integrazione
    def direzione(p):
        g = gradiente(mp, p); ng = np.linalg.norm(g)
        return -g / ng if ng > 1e-12 else None                      # versore di discesa (None: gradiente nullo)
    P = [np.asarray(p0, float)]
    p = P[0]
    for _ in range(20 * N):                                         # limite di sicurezza sul numero di passi
        if np.linalg.norm(p - B) < 2 * h:                           # vicino all'arrivo: si chiude su B
            P.append(B.copy()); return np.array(P)
        k1 = direzione(p)
        k2 = direzione(p + ds / 2 * k1) if k1 is not None else None
        k3 = direzione(p + ds / 2 * k2) if k2 is not None else None
        k4 = direzione(p + ds * k3) if k3 is not None else None
        if k4 is None:
            raise SystemExit(f"RK4: gradiente nullo in {np.round(p, 3)}, il percorso non prosegue")
        q = p + ds / 6 * (k1 + 2 * k2 + 2 * k3 + k4)                 # passo RK4
        if mp["peso"](q[::-1][None])[0] < 0.999:
            raise SystemExit(f"RK4: il passo da {np.round(p, 3)} entrerebbe nel margine di un ostacolo")
        P.append(q); p = q
    raise SystemExit(f"RK4: B non raggiunto in {20 * N} passi (ultimo punto {np.round(p, 3)})")
def orario(mp, perc, t0):                                           # punti fitti del percorso e istante [s] di arrivo in ognuno
    s = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(perc, axis=0), axis=1))])
    sf = np.linspace(0, s[-1], max(int(s[-1] / (h / 2)), 2))
    Pf = np.stack([np.interp(sf, s, perc[:, c]) for c in range(3)], axis=1)
    dt = mp["F_i"](((Pf[:-1] + Pf[1:]) / 2)[:, ::-1]) * np.diff(sf) * SEC_PER_U    # F * ds, in secondi
    return Pf, t0 + np.concatenate([[0], np.cumsum(dt)])

def durante_il_ricalcolo(Pf, tf, vf, k, t_ev, pos, v0, t_pronto, mp_nuova):
    # Mentre ricalcola il drone continua sul percorso vecchio (punti k, k+1, ...), che passa ancora dalla porta chiusa.
    # Se il prossimo punto e' nel nuovo ostacolo (o nel suo margine) frena (frenata di emergenza) e resta fermo fino a t_pronto.
    # Restituisce i punti volati, l'intervallo da fermo (o None) e se il drone e' arrivato in B prima del percorso nuovo
    libero = mp_nuova["peso"](Pf[:, ::-1]) > 0.999                 # punti del percorso vecchio liberi anche nella mappa nuova
    T, P, V, tl, pl, vl = [], [], [], t_ev, pos, v0
    for i in range(k, len(Pf)):
        if tf[i] >= t_pronto:                                       # il percorso nuovo e' pronto prima di arrivare al punto i
            a = (t_pronto - tl) / (tf[i] - tl)
            return T + [t_pronto], P + [pl + a * (Pf[i] - pl)], V + [vl + a * (vf[i] - vl)], None, False
        if not libero[i]:                                           # il punto i e' nel nuovo ostacolo: il drone si ferma dov'e'
            return T + [min(tl + 1e-3, t_pronto), t_pronto], P + [pl, pl], V + [0.0, 0.0], (tl, t_pronto), False
        T.append(tf[i]); P.append(Pf[i]); V.append(vf[i]); tl, pl, vl = tf[i], Pf[i], vf[i]
    return T, P, V, None, True                                      # arrivato in B mentre ricalcolava

# 3) VOLO: ogni drone vola nella sua copia della stanza; la porta m si chiude quando QUEL drone supera x = X_CAMBIO[m].
# Al cambio il drone ricalcola per il tempo del suo metodo continuando sul percorso vecchio, poi riparte sulla mappa nuova
def vola(metodo):
    T, P, V, piani, fermo, ricalcolo, cambi = [0.0], [A.copy()], [None], [], [], [], []
    pos, t, m = A.copy(), 0.0, 0
    while True:
        Pf, tf = orario(mappe[m], percorso(mappe[m], pos), t)
        vf = V_MAX / mappe[m]["F_i"](Pf[:, ::-1])                 # velocita' [m/s] in ogni punto: v = V_MAX / F
        piani.append((t, Pf, tf))
        if V[-1] is None: V[-1] = vf[0]
        nuova = m + 1 if (m + 1) in X_CAMBIO else None
        oltre = Pf[:, 0] > X_CAMBIO[nuova] if nuova else np.zeros(len(Pf), bool)
        if not oltre.any():                                         # nessun altro cambio: vola fino a B
            T += list(tf[1:]); P += list(Pf[1:]); V += list(vf[1:])
            return dict(T=np.array(T), P=np.array(P), V=np.array(V), piani=piani, fermo=fermo, ricalcolo=ricalcolo, cambi=cambi)
        k = max(int(np.argmax(oltre)), 1)                           # il punto in cui il drone supera X_CAMBIO: la porta si chiude
        T += list(tf[1:k + 1]); P += list(Pf[1:k + 1]); V += list(vf[1:k + 1])
        t_ev, t_pronto = tf[k], tf[k] + tempi[nuova][metodo]        # la porta si chiude; quando il percorso nuovo e' pronto
        ricalcolo.append((t_ev, t_pronto)); cambi.append(t_ev)
        Tr, Pr, Vr, fr, arrivato = durante_il_ricalcolo(Pf, tf, vf, k + 1, t_ev, Pf[k], vf[k], t_pronto, mappe[nuova])
        T += Tr; P += Pr; V += Vr
        if fr: fermo.append(fr)
        if arrivato:
            return dict(T=np.array(T), P=np.array(P), V=np.array(V), piani=piani, fermo=fermo, ricalcolo=ricalcolo, cambi=cambi)
        pos, t, m = np.array(P[-1]), t_pronto, nuova
T0 = orario(mappe[0], percorso(mappe[0], A), 0.0)[1][-1]           # durata del volo se le porte restassero aperte
voli = {met: vola(met) for met in METODI}
print(f"volo con le porte aperte: {T0:.2f} s")
for met, v in voli.items():
    ric, fer = [b - a for a, b in v["ricalcolo"]], sum(b - a for a, b in v["fermo"])
    print(f"{met:9s}: porte chiuse a t = {', '.join(f'{c:.2f}' for c in v['cambi'])} s; arrivo in B dopo {v['T'][-1]:6.2f} s; "
          f"ricalcola per {sum(ric):6.3f} s ({' + '.join(f'{a * 1000:.1f} ms' for a in ric)}), di cui {fer:6.3f} s fermo davanti a una porta chiusa")
for met in METODI:
    if met != "Block FIM":
        print(f"il drone del {met} arriva {voli[met]['T'][-1] - voli['Block FIM']['T'][-1]:.2f} s dopo quello del Block FIM")

In [ ]:
# RIPIANIFICAZIONE NEL LABIRINTO - GRAFICA: animazione 3D dei droni e grafico delle velocita' (usa i voli calcolati nella cella sopra).
# Si cambia solo per l'aspetto: colori, dimensioni, velocita' dell'animazione.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 4) PARAMETRI DELLA GRAFICA E GEOMETRIA DA DISEGNARE
DT_FRAME = 0.3                                                      # un fotogramma ogni 0.3 s di volo
MS_FRAME = 60                                                       # il Play mostra un fotogramma ogni 60 ms: animazione 0.3 / 0.06 = 5 volte piu' veloce del vero
GRIGIO, ARANCIO, BLU, ROSSO = "#52514e", "#eb6834", "#2a78d6", "#c0392b"   # FMM, FIM-Jacobi, Block FIM (come nella stanza); porte chiuse
colore = {"FMM": GRIGIO, "FIM-Jacobi": ARANCIO, "Block FIM": BLU}
chiaro = {"FMM": "#b9b8b4", "FIM-Jacobi": "#f7c3a8", "Block FIM": "#a9cbf0"}   # sfumatura chiara dello stesso colore (lampeggio)
scena_di = {met: "scene" if i == 0 else f"scene{i + 1}" for i, met in enumerate(METODI)}   # una scena per drone, da sinistra
DIM_DRONE = 4                                                       # diametro del pallino del drone e dei punti A e B [pixel]

def mesh(ostacoli, nome, col, opacita):                              # un solo Mesh3d con tutti i parallelepipedi
    X, Y, Z, I, J, K = [], [], [], [], [], []
    for b in ostacoli:
        o = len(X)
        X += [b[0], b[1], b[1], b[0], b[0], b[1], b[1], b[0]]; Y += [b[2], b[2], b[3], b[3]] * 2; Z += [b[4]] * 4 + [b[5]] * 4
        for i, j, k in [(0, 1, 2), (0, 2, 3), (4, 5, 6), (4, 6, 7), (0, 1, 5), (0, 5, 4), (1, 2, 6), (1, 6, 5), (2, 3, 7), (2, 7, 6), (3, 0, 4), (3, 4, 7)]:
            I.append(o + i); J.append(o + j); K.append(o + k)
    return go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color=col, opacity=opacita, name=nome, flatshading=True, hoverinfo="name")
a0, a1, a2 = (mappe[m]["amb"] for m in range(3))
porta1 = [f for f in a0["FORO"] if f not in a1["FORO"]][0]          # il foro che sparisce nella mappa 1
porta2 = [f for f in a1["FORO"] if f not in a2["FORO"]][0]          # il foro che sparisce nella mappa 2
interni = [b for b in a0["ZONA"] if b[0] > -0.99 and b[1] < 0.99]   # i muri interni (le pareti della stanza no: c'e' il contorno)
fori_di = lambda b: [f for f in a0["FORO"] if b[0] - 1e-9 <= f[0] and f[1] <= b[1] + 1e-9]   # le aperture di quel muro
def superficie(b, fori, nome, col, opacita):                       # solo le facce esterne del muro b con i suoi fori (niente facce interne)
    tagli = [sorted({b[2 * a], b[2 * a + 1]} | {min(max(f[2 * a + s], b[2 * a]), b[2 * a + 1]) for f in fori for s in (0, 1)}) for a in range(3)]
    n = [len(t) - 1 for t in tagli]
    pieno = np.zeros(n, bool)                                       # cella (ix, iy, iz) del muro: piena se non sta in un foro
    for ix in range(n[0]):
        for iy in range(n[1]):
            for iz in range(n[2]):
                c = [(tagli[a][i] + tagli[a][i + 1]) / 2 for a, i in enumerate((ix, iy, iz))]
                pieno[ix, iy, iz] = not any(f[0] <= c[0] <= f[1] and f[2] <= c[1] <= f[3] and f[4] <= c[2] <= f[5] for f in fori)
    X, Y, Z, I, J, K = [], [], [], [], [], []
    for ix, iy, iz in zip(*np.nonzero(pieno)):
        for asse in range(3):
            for verso in (0, 1):                                    # faccia verso il basso (0) o verso l'alto (1) lungo l'asse
                vic = [ix, iy, iz]; vic[asse] += 1 if verso else -1
                if 0 <= vic[asse] < n[asse] and pieno[tuple(vic)]: continue   # confina con un'altra cella piena: faccia interna
                lo = [tagli[a][i] for a, i in enumerate((ix, iy, iz))]; hi = [tagli[a][i + 1] for a, i in enumerate((ix, iy, iz))]
                q = lo[asse] if verso == 0 else hi[asse]
                u_, w_ = [a for a in range(3) if a != asse]
                o = len(X)
                for du, dw in [(0, 0), (1, 0), (1, 1), (0, 1)]:
                    v = [0.0] * 3; v[asse] = q; v[u_] = (lo[u_], hi[u_])[du]; v[w_] = (lo[w_], hi[w_])[dw]
                    X.append(v[0]); Y.append(v[1]); Z.append(v[2])
                I += [o, o]; J += [o + 1, o + 2]; K += [o + 2, o + 3]
    return go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color=col, opacity=opacita, name=nome, flatshading=True, hoverinfo="name")
def muri_mesh():                                                    # tutti i muri interni in un solo Mesh3d
    parti = [superficie(b, fori_di(b), "muri", "#9a9a9a", 0.35) for b in interni]
    X, Y, Z, I, J, K = [], [], [], [], [], []
    for m_ in parti:
        o = len(X); X += list(m_.x); Y += list(m_.y); Z += list(m_.z)
        I += [o + i for i in m_.i]; J += [o + j for j in m_.j]; K += [o + k for k in m_.k]
    return go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color="#9a9a9a", opacity=0.35, name="muri", flatshading=True, hoverinfo="name")
spigoli = []                                                        # i 12 spigoli della stanza
for a in (-1, 1):
    for b in (-1, 1):
        spigoli += [[(-1, a, b), (1, a, b)], [(a, -1, b), (a, 1, b)], [(a, b, -1), (a, b, 1)]]
stanza = go.Scatter3d(x=sum([[p[0][0], p[1][0], None] for p in spigoli], []), y=sum([[p[0][1], p[1][1], None] for p in spigoli], []),
                      z=sum([[p[0][2], p[1][2], None] for p in spigoli], []), mode="lines", line=dict(color="gray", width=2),
                      name="stanza (20 m)", hoverinfo="skip")
ab = go.Scatter3d(x=[A[0], B[0]], y=[A[1], B[1]], z=[A[2], B[2]], mode="markers+text", text=["  A (partenza)", "  B (arrivo)"],
                  marker=dict(size=DIM_DRONE, color=["green", "black"]), textfont=dict(size=14), name="A e B")
def centro(b):                                                      # dove scrivere il nome di un ostacolo: sopra il centro
    return [(b[0] + b[1]) / 2], [(b[2] + b[3]) / 2], [b[5] + 0.08]

# 5) ANIMAZIONE 3D: da sinistra i droni del FMM, del FIM-Jacobi e del Block FIM (stessi istanti)
def compare(b, t_ev, t, nome, scena):                               # porta che si chiude all'istante t_ev (prima: vuoto)
    if t < t_ev:
        return [go.Mesh3d(x=[], y=[], z=[], i=[], j=[], k=[], name=nome, scene=scena, showlegend=False),
                go.Scatter3d(x=[], y=[], z=[], mode="text", name=nome, scene=scena, showlegend=False)]
    cx, cy, cz = centro(b)
    mm = mesh([b], nome, ROSSO, 0.85); mm.update(scene=scena, showlegend=False)
    return [mm, go.Scatter3d(x=cx, y=cy, z=cz, mode="text", text=[nome], textfont=dict(size=13, color=ROSSO), scene=scena, showlegend=False)]
def posizione(met, t):
    v = voli[met]; T, P, V = v["T"], v["P"], v["V"]
    if t >= T[-1]: return P[-1], 0.0
    k = np.searchsorted(T, t, side="right"); a = (t - T[k - 1]) / max(T[k] - T[k - 1], 1e-12)
    return P[k - 1] + a * (P[k] - P[k - 1]), V[k - 1] + a * (V[k] - V[k - 1])
def drone(met, t):                                                  # scia colorata con la velocita', percorso previsto e drone
    v = voli[met]; T, P, V = v["T"], v["P"], v["V"]
    pos, vel = posizione(met, t)
    fermo = any(a <= t < b for a, b in v["fermo"])
    ricalcola = any(a <= t < b for a, b in v["ricalcolo"])          # mentre ricalcola il pallino lampeggia
    piano = [p for p in v["piani"] if p[0] <= t][-1]
    resto = np.vstack([pos[None], piano[1][piano[2] > t]]) if not fermo and t < T[-1] else np.empty((0, 3))
    scia, vs = np.vstack([P[T <= t][::3], pos[None]]), np.append(V[T <= t][::3], vel)
    sc, c = scena_di[met], colore[met]
    return [go.Scatter3d(x=scia[:, 0], y=scia[:, 1], z=scia[:, 2], mode="lines", scene=sc, showlegend=False, name=f"scia {met}",
                         line=dict(color=vs, colorscale="Turbo", cmin=0.0, cmax=V_MAX, width=12,
                                   colorbar=dict(title="velocita'<br>[m/s]", x=1.0, len=0.6) if met == "Block FIM" else None,
                                   showscale=(met == "Block FIM"))),
            go.Scatter3d(x=resto[:, 0], y=resto[:, 1], z=resto[:, 2], mode="lines", scene=sc, showlegend=False,
                         line=dict(color="black", width=2, dash="dash"), name=f"percorso previsto {met}"),
            go.Scatter3d(x=[pos[0]], y=[pos[1]], z=[pos[2]], mode="markers", scene=sc, showlegend=False, name=f"drone {met}",
                         marker=dict(size=DIM_DRONE, color=chiaro[met] if ricalcola and round(t / DT_FRAME) % 2 else c,   # lampeggia: un fotogramma si' e uno no
                         line=dict(color=c, width=2)))]
def stato(met, t):
    v = voli[met]
    if t >= v["T"][-1]: return "arrivato"
    if any(a <= t < b for a, b in v["fermo"]): return "FERMO davanti alla porta, ricalcola"
    if any(a <= t < b for a, b in v["ricalcolo"]): return f"ricalcola, vola sul percorso vecchio a {posizione(met, t)[1]:.1f} m/s"
    return f"in volo a {posizione(met, t)[1]:.1f} m/s"
ASSI = dict(range=[-1.0, 1.0], autorange=False, showticklabels=False, title="")
SCENA = dict(xaxis=ASSI, yaxis=ASSI, zaxis=ASSI, aspectmode="cube", camera=dict(eye=dict(x=1.2, y=-1.7, z=1.3)))
def statiche(scena):
    out = []
    for tr in [stanza, muri_mesh(), ab]:
        tr = go.Figure(tr).data[0]; tr.update(scene=scena, showlegend=False); out.append(tr)
    return out
def dati(t):
    out = []
    for met in METODI:
        sc = scena_di[met]
        c = voli[met]["cambi"] + [np.inf] * 2
        out += compare(porta1, c[0], t, "porta 1", sc) + compare(porta2, c[1], t, "porta 2", sc) + drone(met, t)
    return out
def titolo(t):
    return f"t = {t:4.1f} s   |   " + "   |   ".join(f"<span style='color:{colore[m]}'>{m}: {stato(m, t)}</span>" for m in METODI)
t_fine = max(v["T"][-1] for v in voli.values())
istanti = np.append(np.arange(0.0, t_fine, DT_FRAME), t_fine)
fisse = sum((statiche(scena_di[m]) for m in METODI), [])
mobili = list(range(len(fisse), len(fisse) + len(dati(0.0))))
dove = {"FMM": "CPU", "FIM-Jacobi": "GPU", "Block FIM": "GPU"}
fig = make_subplots(rows=1, cols=len(METODI), specs=[[{"type": "scene"}] * len(METODI)], horizontal_spacing=0.01,
                    subplot_titles=[f"{m} ({dove[m]}): arriva a {voli[m]['T'][-1]:.1f} s" for m in METODI])
for tr in fisse + dati(0.0): fig.add_trace(tr)
fig.frames = [go.Frame(data=dati(t), traces=mobili, name=f"{t:.2f}", layout=dict(title=dict(text=titolo(t)))) for t in istanti]
fig.update_layout(title=dict(text=titolo(0.0), x=0.5), **{sc: SCENA for sc in scena_di.values()}, height=920, margin=dict(l=0, r=0, t=80, b=290),
                  updatemenus=[dict(type="buttons", direction="left", x=0.08, y=0.0, xanchor="right", yanchor="top", pad=dict(r=10, t=60), buttons=[
                      dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=MS_FRAME, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                      dict(label="Pausa", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
                  sliders=[dict(x=0.1, len=0.88, y=0.0, yanchor="top", currentvalue=dict(prefix="tempo di volo t = ", suffix=" s"), pad=dict(t=40),
                                steps=[dict(label=f"{t:.0f}" if abs(t - round(t)) < 1e-9 else "", method="animate",
                                            args=[[f"{t:.2f}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for t in istanti])])
# Riquadro sotto l'animazione: come si passa dalla u del FM2 a metri e secondi
uA = mappe[0]["u"][nodo(A)]                                         # u(A) con le porte aperte
spessore = (interni[0][1] - interni[0][0]) * METRI_PER_UNITA        # spessore di un muro interno [m]
fig.add_annotation(xref="paper", yref="paper", x=0.5, y=-0.27, xanchor="center", yanchor="top", align="left", showarrow=False,
                   bgcolor="#f4f6fa", bordercolor="#9a9a9a", borderwidth=1, borderpad=8, font=dict(size=13),
                   text=(f"<b>Dalla u del FM2 a metri e secondi</b><br>"
                         f"Scala: 1 unita' = {METRI_PER_UNITA:.0f} m, quindi la stanza [-1, 1]^3 e' {2 * METRI_PER_UNITA:.0f} x {2 * METRI_PER_UNITA:.0f} x {2 * METRI_PER_UNITA:.0f} m "
                         f"(muri spessi {spessore:.0f} m, margine di sicurezza {mappe[0]['amb']['MARGINE'] * METRI_PER_UNITA * 100:.0f} cm)<br>"
                         f"u = somma di F x tratto percorso: una distanza pesata (F = 1 lontano dagli ostacoli, F = 10 rasente)<br>"
                         f"Velocita' massima V_MAX = {V_MAX:.0f} m/s dove F = 1: tempo = u x {METRI_PER_UNITA:.0f} m / {V_MAX:.0f} m/s = u x {SEC_PER_U:.0f} s; "
                         f"velocita' v = V_MAX / F (da {V_MAX / 10:.1f} a {V_MAX:.0f} m/s)<br>"
                         f"Esempio: porte aperte, u(A) = {uA:.2f}, quindi {uA:.2f} x {SEC_PER_U:.0f} s = {uA * SEC_PER_U:.1f} s di volo | "
                         f"Play: 1 fotogramma = {DT_FRAME} s di volo ogni {MS_FRAME} ms, cioe' {DT_FRAME * 1000 / MS_FRAME:.0f} volte piu' veloce del vero"))
fig.write_html("ripianificazione_affiancata.html", include_plotlyjs="cdn")    # anche come file da scaricare: si apre nel browser
fig.show()

# 6) VELOCITA' NEL TEMPO e TEMPO A VELOCITA' MASSIMA dei tre droni
g = make_subplots(rows=1, cols=2, column_widths=[0.68, 0.32], horizontal_spacing=0.1,
                  subplot_titles=("Velocita' nel tempo (fasce = ricalcola; linee = una porta si chiude)", "Come passa il tempo di volo"))
quote = {}
for met, v in voli.items():
    T, V = v["T"], v["V"]
    g.add_trace(go.Scatter(x=T, y=V, mode="lines", line=dict(color=colore[met], width=2.5), name=f"{met}: arriva a {T[-1]:.1f} s"), row=1, col=1)
    for a, b in v["ricalcolo"]:
        g.add_vrect(x0=a, x1=max(b, a + 0.05), fillcolor=colore[met], opacity=0.2, line_width=0, row=1, col=1)
    dt, vm = np.diff(T), (V[:-1] + V[1:]) / 2
    quote[met] = {"a velocita' massima": dt[vm >= V_PIENA].sum(), "piu' lento": dt[(vm > 0.01) & (vm < V_PIENA)].sum(),
                  "fermo davanti alla porta": dt[vm <= 0.01].sum()}
for met, v in voli.items():
    for i, t_ev in enumerate(v["cambi"]):
        g.add_vline(x=t_ev, line=dict(color=colore[met], dash="dot"), annotation_text=f"porta {i + 1}" if met == "Block FIM" else None,
                    annotation_position="top left", row=1, col=1)
g.add_hline(y=V_MAX, line=dict(color="gray", dash="dash"), annotation_text="V_MAX", row=1, col=1)
for voce, col in [("a velocita' massima", "#2ca02c"), ("piu' lento", "#bcbd22"), ("fermo davanti alla porta", "#7f7f7f")]:
    g.add_trace(go.Bar(x=list(quote), y=[quote[m][voce] for m in quote], name=voce, marker_color=col,
                       text=[f"{quote[m][voce]:.1f} s" for m in quote], textposition="inside"), row=1, col=2)
g.update_layout(template="plotly_white", barmode="stack", height=430, legend=dict(orientation="h", y=-0.2),
                title="Arrivo in B: " + ", ".join(f"{met} {voli[met]['T'][-1]:.1f} s ({quote[met]['fermo davanti alla porta']:.1f} s fermo)" for met in METODI))
g.update_xaxes(title_text="tempo di volo [s]", row=1, col=1); g.update_yaxes(title_text="velocita' [m/s]", row=1, col=1)
g.update_yaxes(title_text="tempo [s]", row=1, col=2)
g.write_html("ripianificazione_velocita_affiancata.html", include_plotlyjs="cdn")
g.show()
for met, q in quote.items():
    print(f"{met:9s}: " + " | ".join(f"{k} {v:5.1f} s" for k, v in q.items()))